In [1]:
!pip install -q sentence-transformers catboost --upgrade

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torch

2025-09-21 16:54:21.537268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758473661.788936     239 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758473661.871198     239 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [55]:
items = pd.read_parquet("/kaggle/input/path-to-zen/Archive/items.parquet")
train = pd.read_csv("/kaggle/input/path-to-zen/Archive/train.csv")

The notebook kept crashing because it ran out of memory, so I want to free memory along the way.

In [4]:
import gc

Symmetrizing the pairs as an augmentation improved the score.

In [5]:
# удаляем мусор
train_tmp = train[~train['target'].isin(['doubles', 'unavailable'])]

# Делаем копию с переставленными left/right
train_swapped = train_tmp.rename(columns={"leftItemId": "rightItemId", "rightItemId": "leftItemId"})

# Объединяем
train_tmp = pd.concat([train_tmp, train_swapped], ignore_index=True)

In [6]:
# фильтрация
train_tmp = train_tmp[train_tmp['target'].isin(['relevant_minus','relevant','no_relevant','relevant_plus'])]

#таргет в число
label2id = {l:i for i,l in enumerate(["no_relevant", "relevant_minus", "relevant", "relevant_plus"])}
id2label = {v:k for k,v in label2id.items()}
train_tmp['target_id'] = train_tmp['target'].map(label2id)
left = items.rename(columns={c: c+"_left" for c in items.columns})
right = items.rename(columns={c: c+"_right" for c in items.columns})

df = train_tmp.merge(left, left_on='leftItemId', right_on='itemId_left', how='left')
df = df.merge(right, left_on='rightItemId', right_on='itemId_right', how='left')

#добавляем текст в датасет, обрезаем на первые 500
MAX_CONTENT = 500
df['text_left'] = df['title_left'].fillna('') + " " + df['content_left'].fillna('').str[:MAX_CONTENT]
df['text_right'] = df['title_right'].fillna('') + " " + df['content_right'].fillna('').str[:MAX_CONTENT]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [27]:
del train_tmp
gc.collect()

40

In [8]:
df.head()

,leftItemId,rightItemId,target,target_id,authorId_left,content_left,itemId_left,title_left,authorId_right,content_right,itemId_right,title_right,text_left,text_right
0,8456188648456666255,17842617327504664637,relevant_minus,1,7017069484959831201,Глава Ореховской ОПГ и один из самых влиятельн...,8456188648456666255,За что авторитет Сильвестр-Тимофеев не пожалел...,7017069484959831201,"Вроде бы, россияне уже имеют одно светило, кот...",17842617327504664637,Несколько фактов об Александре Розенбауме,За что авторитет Сильвестр-Тимофеев не пожалел...,Несколько фактов об Александре Розенбауме Врод...
1,2587705644018673303,4730919752548875048,relevant_minus,1,9820097495407617467,Дизайнер Екатерина Бубнова продумала комфортну...,2587705644018673303,Всего на 33 кв. м дизайнер оформила полноценну...,4170089941237414359,"Какую столешницу выбрать: деревянную, каменную...",4730919752548875048,"Столешница на кухню, что выбрать: МДФ, ДСП, де...",Всего на 33 кв. м дизайнер оформила полноценну...,"Столешница на кухню, что выбрать: МДФ, ДСП, де..."
2,14440570074797857092,9007102992088629866,relevant,2,16414771911870906244,"Чтобы легально заниматься бизнесом, необходи...",14440570074797857092,Три простых шага по открытию ИП,8349252821380888391,Систематическое занятие какой-либо деятельност...,9007102992088629866,Как самостоятельно открыть ИП?,Три простых шага по открытию ИП Чтобы легаль...,Как самостоятельно открыть ИП? Систематическое...
3,10768915202093280757,8382562539112058142,relevant,2,11876287328313618044,"Добрый день, уважаемые гости и постоянные чит...",10768915202093280757,Отличное решение вместо гипсокартона и натяжно...,13816308203894807670,Всем привет. Недавно мы выкладывали на канал ...,8382562539112058142,"Минусы кровли из профлиста, которые мы заметил...",Отличное решение вместо гипсокартона и натяжно...,"Минусы кровли из профлиста, которые мы заметил..."
4,2799154957886243971,11333177839206244806,no_relevant,0,3030333479288115002,В этом году в конце ноября муж предложил съезд...,2799154957886243971,Поехала с мужем на Волгу порыбачить в конце но...,5532370162254990738,С опытом путешествий всё чаще самые мощные эмо...,11333177839206244806,Познакомился на берегу Баренцева моря с рыбако...,Поехала с мужем на Волгу порыбачить в конце но...,Познакомился на берегу Баренцева моря с рыбако...


In [9]:
# решил использовать легкую, так как остальные это просто ад ждать
MODEL_TO_USE = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_TO_USE, device=device)

In [ ]:
if (MODEL_TO_USE == "paraphrase-multilingual-MiniLM-L12-v2"):
    emb_left = model.encode(df['text_left'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)
    emb_right = model.encode(df['text_right'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)
else:
    emb_left = pd.read_parquet("/kaggle/input/path-to-zen/emb_left.parquet")
    emb_right = pd.read_parquet("/kaggle/input/path-to-zen/emb_right.parquet")

Batches:   0%|          | 0/6516 [00:00<?, ?it/s]

Batches:   0%|          | 0/6516 [00:00<?, ?it/s]

In [14]:
np.save("emb_left.npy", emb_left)
np.save("emb_right.npy", emb_right)

In [15]:
emb_left_pd = pd.DataFrame(emb_left)

emb_left_pd.to_parquet("emb_left.parquet", index=False)

In [16]:
emb_right_pd = pd.DataFrame(emb_right)

emb_right_pd.to_parquet("emb_right.parquet", index=False)

In [28]:
del emb_right_pd
del emb_left_pd
gc.collect()

0

In [18]:
#добавим всякого
X = np.hstack([emb_left, emb_right, np.abs(emb_left-emb_right), emb_left*emb_right])
y = df['target_id'].values

In [ ]:
"""
подбор гиперпараметров
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

for depth in [6, 8, 10]:
    for lr in [0.03, 0.05, 0.1]:
        model_cb = CatBoostClassifier(
            iterations=2000,
            depth=depth,
            learning_rate=lr,
            loss_function='MultiClass',
            eval_metric='TotalF1',
            task_type='GPU',
            early_stopping_rounds=100,
            verbose=False
        )
        model_cb.fit(X_train, y_train, eval_set=(X_val, y_val))
        preds = model_cb.predict(X_val)
        f1 = f1_score(y_val, preds, average='weighted')
        print(f"depth={depth}, lr={lr} → F1={f1:.4f}")"""

In [20]:
#катбуст
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

#считаем веса, чтобы учитывать все классы
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {i:w for i,w in zip(classes, class_weights)}

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model_cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.1,
    depth=8,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Weighted',
    task_type='GPU' if device=='cuda' else 'CPU',
    class_weights=class_weights_dict,
    random_seed=42,
    verbose=100
)



model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)


# валидация
y_pred = model_cb.predict(val_pool).astype(int).flatten()
f1 = f1_score(y_val, y_pred, average='weighted')
print(f"Weighted F1 на валидации: {f1:.4f}")

0:	learn: 0.3444794	test: 0.3424942	best: 0.3424942 (0)	total: 532ms	remaining: 26m 34s
100:	learn: 0.5099609	test: 0.4764443	best: 0.4764443 (100)	total: 15.2s	remaining: 7m 16s
200:	learn: 0.5526700	test: 0.5014663	best: 0.5014663 (200)	total: 29.7s	remaining: 6m 53s
300:	learn: 0.5861834	test: 0.5169656	best: 0.5169669 (296)	total: 44s	remaining: 6m 34s
400:	learn: 0.6161908	test: 0.5307077	best: 0.5309780 (398)	total: 57.8s	remaining: 6m 14s
500:	learn: 0.6435738	test: 0.5422060	best: 0.5426288 (497)	total: 1m 11s	remaining: 5m 57s
600:	learn: 0.6702538	test: 0.5550208	best: 0.5550208 (600)	total: 1m 25s	remaining: 5m 42s
700:	learn: 0.6930893	test: 0.5644179	best: 0.5645661 (699)	total: 1m 39s	remaining: 5m 27s
800:	learn: 0.7168730	test: 0.5753997	best: 0.5756690 (797)	total: 1m 54s	remaining: 5m 13s
900:	learn: 0.7373532	test: 0.5858519	best: 0.5858519 (900)	total: 2m 8s	remaining: 5m
1000:	learn: 0.7564699	test: 0.5960325	best: 0.5960325 (1000)	total: 2m 22s	remaining: 4m 45s
1

Since there are many duplicates (left-right and right-left), there was effectively a leak, but overall it is fine.

In [21]:
model = SentenceTransformer(MODEL_TO_USE, device=device)

In [22]:
# прогон на тесте
test = pd.read_csv("/kaggle/input/path-to-zen/Archive/test.csv")

test_df = test.merge(left, left_on='leftItemId', right_on='itemId_left', how='left')
test_df = test_df.merge(right, left_on='rightItemId', right_on='itemId_right', how='left')

test_df['text_left'] = test_df['title_left'].fillna('') + " " + test_df['content_left'].fillna('').str[:MAX_CONTENT]
test_df['text_right'] = test_df['title_right'].fillna('') + " " + test_df['content_right'].fillna('').str[:MAX_CONTENT]

emb_left_test = model.encode(test_df['text_left'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)
emb_right_test = model.encode(test_df['text_right'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)

# весь тот же препроцессинг

X_test = np.hstack([emb_left_test, emb_right_test, np.abs(emb_left_test-emb_right_test), emb_left_test*emb_right_test])
test_pool = Pool(X_test)

probs1 = model_cb.predict_proba(test_pool)

Batches:   0%|          | 0/807 [00:00<?, ?it/s]

Batches:   0%|          | 0/807 [00:00<?, ?it/s]

In [24]:
test_swapped = test_df.copy()
test_swapped = test_swapped.rename(columns={
    "leftItemId": "rightItemId", 
    "rightItemId": "leftItemId"
})

emb_left_sw = emb_right_test
emb_right_sw = emb_left_test

X_test_swapped = np.hstack([
    emb_left_sw,
    emb_right_sw,
    np.abs(emb_left_sw - emb_right_sw),
    emb_left_sw * emb_right_sw
])

probs2 = model_cb.predict_proba(X_test_swapped)
assert probs2.shape == probs1.shape

# усредняем вероятности, так как мы два раза предсказали
probs_avg = (probs1 + probs2) / 2

test_preds = np.argmax(probs_avg, axis=1)

Then I added a super heavy model to the ensemble that I had used to compute embeddings earlier; I think it took about two hours to produce them.

In [42]:
train_tmp = train

Everything is the same, except the content is cut to 750 characters, because back then I truncated it to 750.

In [57]:
train_tmp = train_tmp[train_tmp['target'].isin(['relevant_minus','relevant','no_relevant','relevant_plus'])]
label2id = {l:i for i,l in enumerate(["no_relevant", "relevant_minus", "relevant", "relevant_plus"])}
id2label = {v:k for k,v in label2id.items()}
train_tmp['target_id'] = train_tmp['target'].map(label2id)
left = items.rename(columns={c: c+"_left" for c in items.columns})
right = items.rename(columns={c: c+"_right" for c in items.columns})


df = train_tmp.merge(left, left_on='leftItemId', right_on='itemId_left', how='left')
df = df.merge(right, left_on='rightItemId', right_on='itemId_right', how='left')

MAX_CONTENT = 500
df['text_left'] = df['title_left'].fillna('') + " " + df['content_left'].fillna('').str[:750]
df['text_right'] = df['title_right'].fillna('') + " " + df['content_right'].fillna('').str[:750]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [47]:
import sys

for name, obj in globals().items():
    try:
        size = sys.getsizeof(obj)
        if size > 10**6:  # фильтруем только >1MB
            print(f"{name:<30} {size / 1024**2:.2f} MB")
    except TypeError:
        pass

items                          2669.26 MB
train                          16.97 MB
train_swapped                  18.34 MB
left                           2669.26 MB
right                          2669.26 MB
df                             5261.93 MB
emb_left                       610.82 MB
emb_right                      610.82 MB
X                              2443.29 MB
X_train                        1954.63 MB
X_val                          488.66 MB
y_train                        1.27 MB
test_df                        1242.63 MB
emb_left_test                  75.64 MB
emb_right_test                 75.64 MB
X_test                         302.55 MB
test_swapped                   1242.63 MB
emb_left_sw                    75.64 MB
emb_right_sw                   75.64 MB
X_test_swapped                 302.55 MB
probs_avg                      1.58 MB
submission                     3.75 MB
_26                            36.68 MB
train_tmp                      19.93 MB


Memory cleanup again, because I really did not want the notebook to crash.

In [56]:
del X
gc.collect()

17

In [51]:
emb_left = pd.read_parquet("/kaggle/input/path-to-zen/emb_left.parquet")
emb_right = pd.read_parquet("/kaggle/input/path-to-zen/emb_right.parquet")
X = np.hstack([emb_left, emb_right, np.abs(emb_left-emb_right), emb_left*emb_right])
y = df['target_id'].values

In [40]:
df.shape

(208494, 14)

In [52]:
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Вычисляем веса классов
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {i:w for i,w in zip(classes, class_weights)}

train_pool = Pool(X_train, y_train)
val_pool = Pool(X_val, y_val)

model_cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.1,
    depth=8,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Weighted',
    task_type='GPU' if device=='cuda' else 'CPU',
    class_weights=class_weights_dict,
    random_seed=42,
    verbose=100
)



model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)

y_pred = model_cb.predict(val_pool).astype(int).flatten()
f1 = f1_score(y_val, y_pred, average='weighted')
print(f"Weighted F1 на валидации: {f1:.4f}")

0:	learn: 0.3491852	test: 0.3449168	best: 0.3449168 (0)	total: 341ms	remaining: 11m 22s
100:	learn: 0.4882446	test: 0.4346479	best: 0.4346479 (100)	total: 23.8s	remaining: 7m 28s
200:	learn: 0.5386994	test: 0.4417715	best: 0.4421182 (191)	total: 46.4s	remaining: 6m 54s
300:	learn: 0.5802961	test: 0.4463653	best: 0.4476050 (274)	total: 1m 8s	remaining: 6m 26s
400:	learn: 0.6176482	test: 0.4482547	best: 0.4490410 (342)	total: 1m 30s	remaining: 6m 1s
500:	learn: 0.6497616	test: 0.4507198	best: 0.4507198 (500)	total: 1m 52s	remaining: 5m 37s
600:	learn: 0.6804323	test: 0.4503503	best: 0.4521899 (560)	total: 2m 15s	remaining: 5m 14s
700:	learn: 0.7062822	test: 0.4521973	best: 0.4523565 (696)	total: 2m 37s	remaining: 4m 51s
800:	learn: 0.7306198	test: 0.4517277	best: 0.4530910 (747)	total: 2m 59s	remaining: 4m 28s
900:	learn: 0.7534950	test: 0.4514623	best: 0.4530910 (747)	total: 3m 21s	remaining: 4m 6s
1000:	learn: 0.7751381	test: 0.4499361	best: 0.4530910 (747)	total: 3m 44s	remaining: 3m 

In [59]:
model = SentenceTransformer('DeepPavlov/rubert-base-cased-sentence', device=device)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [61]:
test_df = test.merge(left, left_on='leftItemId', right_on='itemId_left', how='left')
test_df = test_df.merge(right, left_on='rightItemId', right_on='itemId_right', how='left')

test_df['text_left'] = test_df['title_left'].fillna('') + " " + test_df['content_left'].fillna('').str[:750]
test_df['text_right'] = test_df['title_right'].fillna('') + " " + test_df['content_right'].fillna('').str[:750]

emb_left_test = model.encode(test_df['text_left'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)
emb_right_test = model.encode(test_df['text_right'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True, device=device)

X_test = np.hstack([emb_left_test, emb_right_test, np.abs(emb_left_test-emb_right_test), emb_left_test*emb_right_test])
test_pool = Pool(X_test)

probs_hard = model_cb.predict_proba(test_pool)

Batches:   0%|          | 0/807 [00:00<?, ?it/s]

Batches:   0%|          | 0/807 [00:00<?, ?it/s]

In [63]:
probs_ensemble = 0.4 * probs_hard + 0.6 * probs_avg
test_preds = np.argmax(probs_ensemble, axis=1)

Weighted averaging: the heavier model performed worse.

In [64]:
test_preds_labels = [id2label[i] for i in test_preds]


# Создаём сабмит
submission = pd.DataFrame({
    "Unnamed: 0": test.index,
    "target": test_preds_labels
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv сохранён!")

✅ submission.csv сохранён!
